# OS1 Bootup Sequence Synthesizer
**from *Her* — integral-driven pitch**

| Symbol | Meaning |
|--------|---------|
| `f(x)` | Loading **rate** — how fast the system is loading at position x |
| `F(x)` | Cumulated load — `∫₀ˣ f(t) dt` — how much has loaded up to x |

The audio sweep maps **F(x)** onto pitch:  
- `F(x) = 0` → lowest tone (~180 Hz)  
- `F(x) = max` → highest tone (~1100 Hz)  

So a burst of fast loading sounds like a rapid pitch jump; a slow crawl sounds like a long, flat drone.

In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#   "numpy",
#   "scipy",
#   "matplotlib",
#   "ipywidgets",
#   "sounddevice",
#   "jupyter",
# ]
# ///

In [ ]:
# ── dependencies (uv) ────────────────────────────────────────────────────────
import subprocess, sys

def _uv_install(*packages):
    subprocess.check_call([
        'uv', 'pip', 'install', '--system', '-q', *packages
    ])

try:
    import numpy, scipy, matplotlib, ipywidgets, sounddevice
except ImportError:
    print('Installing via uv...')
    _uv_install('numpy', 'scipy', 'matplotlib', 'ipywidgets', 'sounddevice')

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, signal
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

print('Ready.')

In [ ]:
# ── synthesis engine (don't edit) ────────────────────────────────────────────
SR         = 44100
FREQ_LO    = 180.0    # Hz at F(x) = 0%
FREQ_HI    = 1100.0   # Hz at F(x) = 100%
HARMONICS  = [(1.00, 1.00), (0.45, 2.00), (0.20, 3.00), (0.08, 4.00), (0.04, 5.02)]

def synthesize(F_norm: np.ndarray, duration: float = 3.0) -> np.ndarray:
    """
    F_norm : 1-D array of cumulated load values, normalised to [0, 1].
             Index 0 = start of boot, index -1 = end.
    duration: total audio length in seconds.
    """
    n      = int(SR * duration)
    # Resample F_norm to exactly n audio samples
    x_src  = np.linspace(0, 1, len(F_norm))
    x_dst  = np.linspace(0, 1, n)
    F_re   = np.interp(x_dst, x_src, F_norm)

    # Instantaneous frequency driven by cumulated load
    freq_t = FREQ_LO + F_re * (FREQ_HI - FREQ_LO)

    # Phase via cumulative sum of frequency
    phase  = 2 * np.pi * np.cumsum(freq_t) / SR

    wave   = np.zeros(n)
    for amp, mult in HARMONICS:
        wave += amp * np.sin(phase * mult)

    wave  /= np.max(np.abs(wave)) + 1e-9

    # Amplitude envelope
    atk    = int(0.15 * SR)
    rel    = int(0.30 * SR)
    env    = np.ones(n)
    env[:atk]   = np.linspace(0, 1, atk)
    env[-rel:]  = np.linspace(1, 0, rel)

    # Chunk-wise low-pass that opens as pitch rises
    out    = np.empty_like(wave)
    chunk  = 1024
    for i in range(0, n, chunk):
        sl     = slice(i, min(i + chunk, n))
        fc     = float(np.mean(freq_t[sl]))
        cut    = min(fc * 2.2, 18000) / (SR / 2)
        b, a   = signal.butter(2, cut, btype='low')
        out[sl] = signal.lfilter(b, a, wave[sl])

    return (out * env * 0.85).astype(np.float32)


def compute_integral(f, x):
    """Numerically integrate f over x using the cumulative trapezoid rule."""
    y   = np.array([f(xi) for xi in x])
    F   = integrate.cumulative_trapezoid(y, x, initial=0)
    F_norm = (F - F.min()) / (F.max() - F.min() + 1e-12)
    return y, F, F_norm

print('Engine loaded.')

---
## ✏️  Define your loading-rate function  `f(x)`

Write any function of `x` in `[0, 1]`.  
Examples:
```python
return 1                          # constant rate  → linear sweep
return x                          # accelerating   → pitch rises slowly then fast
return 1 - x                      # decelerating   → pitch rises fast then slow
return np.exp(-5*(x-0.5)**2)      # burst in middle
return np.sin(np.pi * x)**2       # smooth bell
```

In [ ]:
# ── ✏️  YOUR FUNCTION HERE ────────────────────────────────────────────────────
def f(x):
    """Loading rate at position x  (x is in [0, 1])."""
    return 1  # ← change this

# ─────────────────────────────────────────────────────────────────────────────
# resolution of the x-axis (increase for smoother curves)
N_POINTS = 500
DURATION = 3.0   # seconds of audio

x   = np.linspace(0, 1, N_POINTS)
f_y, F_y, F_norm = compute_integral(f, x)

# ── plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_color('#333')

axes[0].plot(x, f_y, color='#7b68ee', lw=2)
axes[0].set_title('f(x)  — loading rate', color='#d8d8f0')
axes[0].set_xlabel('progress →', color='#666')

axes[1].plot(x, F_y, color='#50c878', lw=2)
axes[1].set_title('F(x) = ∫f  — cumulated load', color='#d8d8f0')
axes[1].set_xlabel('progress →', color='#666')

freq_curve = FREQ_LO + F_norm * (FREQ_HI - FREQ_LO)
axes[2].plot(x, freq_curve, color='#f0a050', lw=2)
axes[2].set_title('Pitch (Hz)  — what you hear', color='#d8d8f0')
axes[2].set_xlabel('progress →', color='#666')
axes[2].set_ylabel('Hz', color='#666')

plt.tight_layout()
plt.show()
print(f'f(x) range : [{f_y.min():.3f}, {f_y.max():.3f}]')
print(f'F(x) range : [{F_y.min():.3f}, {F_y.max():.3f}]')

---
## 🔊  Generate & play audio

In [ ]:
wave = synthesize(F_norm, duration=DURATION)
display(Audio(wave, rate=SR, autoplay=True))
print(f'Playing {DURATION:.1f}s sweep driven by F(x).')

---
## 🎚️  Interactive slider — hear any load point

Drag to a percentage and click **Play snapshot** to hear the instantaneous pitch at that load point.

In [ ]:
slider = widgets.IntSlider(
    value=0, min=0, max=100, step=1,
    description='Load %',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='520px')
)
freq_out = widgets.Label(value=f'≈ {FREQ_LO:.0f} Hz')
btn      = widgets.Button(description='▶ Play snapshot',
                          button_style='primary',
                          layout=widgets.Layout(width='140px'))
out_area = widgets.Output()

def _update_label(change):
    pct = change['new'] / 100.0
    # Find the x where F_norm first reaches pct
    idx = np.searchsorted(F_norm, pct)
    idx = np.clip(idx, 0, len(F_norm) - 1)
    hz  = FREQ_LO + F_norm[idx] * (FREQ_HI - FREQ_LO)
    freq_out.value = f'≈ {hz:.0f} Hz'

def _play_snapshot(_):
    pct      = slider.value / 100.0
    idx      = np.clip(np.searchsorted(F_norm, pct), 0, len(F_norm) - 1)
    hz       = FREQ_LO + F_norm[idx] * (FREQ_HI - FREQ_LO)
    snap_dur = 0.55
    n        = int(SR * snap_dur)
    t        = np.linspace(0, snap_dur, n, endpoint=False)
    glide    = hz * 0.025
    freq_t   = hz + np.linspace(-glide, glide, n)
    phase    = 2 * np.pi * np.cumsum(freq_t) / SR
    wave     = sum(a * np.sin(phase * m) for a, m in HARMONICS)
    wave    /= np.max(np.abs(wave)) + 1e-9
    atk, rel = int(0.10 * SR), int(0.18 * SR)
    env      = np.ones(n)
    env[:atk]  = np.linspace(0, 1, atk)
    env[-rel:] = np.linspace(1, 0, rel)
    wave = (wave * env * 0.85).astype(np.float32)
    with out_area:
        clear_output(wait=True)
        display(Audio(wave, rate=SR, autoplay=True))

slider.observe(_update_label, names='value')
btn.on_click(_play_snapshot)

display(widgets.VBox([
    widgets.HBox([slider, freq_out]),
    btn,
    out_area
]))